In [ ]:
print("hello world")

In [1]:
#loading data into dataset
import seaborn as sns
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# 1. Check for missing values and handle them.
# 2. Check for duplicate rows and remove them.
# 3. Inspect the "Sex" column for inconsistencies and standardize values.
# 4. Detect and handle outliers in the "Age" and "Fare" columns.
# 5. Create a new "Family_Size" feature based on "SibSp" and "Parch".
# 6. Convert the "Embarked" column to numerical values using label encoding.
# 7. If a "Ticket_Purchase_Date" column existed, convert it to datetime and extract the year and month.
# 8. Clean the "Name" column by removing titles and converting to lowercase.
# 9. Bin the "Age" column into categories like 'Child', 'Teen', 'Adult', and 'Senior'.

In [2]:
#import libs
import pandas as pd
import numpy as np
import re

1. Missing values

In [ ]:
#checking for missing values
# print(df.isna()) #full display of all columns with missing
print(df.isna().mean() * 100)

survived        0.000000
pclass          0.000000
sex             0.000000
age            19.865320
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.224467
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.216611
embark_town     0.224467
alive           0.000000
alone           0.000000
dtype: float64


In [ ]:
#missing age - set it as the median to fill in the blanks
median_age = df['age'].median() #get the median of the column first
df['age'] = df['age'].fillna(median_age) #fillna method fills in the values with 'median_age'ArithmeticError
print(f"Missing values in 'age' after filling: {df['age'].isna().sum()}") #check it with the same prior check method isna to confirm no missing values and print sum of them
df.head()

Missing values in 'age' after filling: 0


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
#deck has too many missing values, could drop the column or leave as is or flag it?

In [10]:
#embark missing values
mode_town = df['embark_town'].mode()[0] #use mode to find most frequent value in the col
df['embark_town'] = df['embark_town'].fillna(mode_town) #fill missing values with the mode value
print(f"Missing values in 'embark_town' now: {df['embark_town'].isna().sum()}") #check sum of missing values after update

Missing values in 'embark_town' now: 0


2. Duplicate rows

In [11]:
#get dupe count
duplicate_count = df.duplicated().sum() #get the amount of dupe rows
print(f"Number of duplicate rows: {duplicate_count}") #print the sum of dupe rows

Number of duplicate rows: 110


In [13]:
df.drop_duplicates(inplace=True) #drop the dupes- inplace modifies the existing df so i dont have to re-assign it
print(f"New shape after removing duplicates: {df.shape}") #print the amount of rows and columns
duplicate_count = df.duplicated().sum() #get the amount of dupe rows
print(f"Number of duplicate rows: {duplicate_count}") #check again for dupe row count

New shape after removing duplicates: (781, 15)
Number of duplicate rows: 0


3. Inspect the "Sex" column for inconsistencies and standardize values.

In [ ]:
print(df['sex'].unique()) #check the unique values in the col
print(df['sex'].value_counts()) #give a count of each unique val

['male' 'female']
sex
male      488
female    293
Name: count, dtype: int64


In [ ]:
#only male, female as values. Looks fine

4. Detect and handle outliers in the "Age" and "Fare" columns.

In [16]:
# Function to calculate bounds
def get_outlier_bounds(column): 
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound
#this creating a function to be used for both the age and fare check
#it'll get the lower/upper bound based on the quanitile

# Check Age
age_lower, age_upper = get_outlier_bounds(df['age'])
age_outliers = df[(df['age'] < age_lower) | (df['age'] > age_upper)]

# Check Fare
fare_lower, fare_upper = get_outlier_bounds(df['fare'])
fare_outliers = df[(df['fare'] < fare_lower) | (df['fare'] > fare_upper)]

print(f"Age outliers detected: {len(age_outliers)}") #print the value
print(f"Fare outliers detected: {len(fare_outliers)}") #print the value

Age outliers detected: 39
Fare outliers detected: 102


In [17]:

df['age'] = df['age'].clip(lower=age_lower, upper=age_upper)
#cap any ages lower than the lower/upper

df['fare'] = df['fare'].clip(lower=fare_lower, upper=fare_upper)

#print check the new value
print(f"New Max Age: {df['age'].max()}")
print(f"New Max Fare: {df['fare'].max()}")

New Max Age: 57.0
New Max Fare: 72.977


5. Create a new "Family_Size" feature based on "SibSp" and "Parch".

In [18]:
# Create Family_Size: SibSp + Parch + 1 (for the passenger)
df['family_size'] = df['sibsp'] + df['parch'] + 1

# Inspect the new column
print(df[['sibsp', 'parch', 'family_size']].head())

# Optional: See the distribution of family sizes
print(df['family_size'].value_counts())

   sibsp  parch  family_size
0      1      0            2
1      1      0            2
2      0      0            1
3      1      0            2
4      0      0            1
family_size
1     443
2     154
3     101
4      28
6      22
5      13
7      12
8       6
11      2
Name: count, dtype: int64


6. Convert the "Embarked" column to numerical values using label encoding.

In [19]:
from sklearn.preprocessing import LabelEncoder

# 1. Ensure 'embarked' has no missing values (using the mode like we did before)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# 2. Initialize the LabelEncoder
le = LabelEncoder()

# 3. Fit and transform the column
df['embarked'] = le.fit_transform(df['embarked'])

# Verify the changes
print(df[['embarked']].head())
print(f"Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

   embarked
0         2
1         0
2         2
3         2
4         2
Mapping: {'C': 0, 'Q': 1, 'S': 2}


^ LabelEncoder usually assigns numbers alphabetically. In this case, C (Cherbourg) becomes 0, Q (Queenstown) becomes 1, and S (Southampton) becomes 2. 

7. Ticket_Purchase_Date" column existed, convert it to datetime and extract the year and month.

In [ ]:
import pandas as pd
import numpy as np

# 1. Create a dummy date column so the code doesn't crash
# We'll pretend everyone bought their tickets in early 1912
df['Ticket_Purchase_Date'] = pd.to_datetime(['1912-01-15'] * len(df))
#this col doesn't exist so is just false data added to demonstrate ^

# 2. Now run your extraction code
df['Purchase_Year'] = df['Ticket_Purchase_Date'].dt.year #get just the year
df['Purchase_Month'] = df['Ticket_Purchase_Date'].dt.month #get just the month

# 3. Check the result
print(df[['Ticket_Purchase_Date', 'Purchase_Year', 'Purchase_Month']].head())

  Ticket_Purchase_Date  Purchase_Year  Purchase_Month
0           1912-01-15           1912               1
1           1912-01-15           1912               1
2           1912-01-15           1912               1
3           1912-01-15           1912               1
4           1912-01-15           1912               1


8. Clean the "Name" column by removing titles and converting to lowercase.

In [ ]:
#col doesn't exist so not sure how to provide an example

In [23]:
bins = [0, 12, 19, 60, 100] #set values for bins
labels = ['Child', 'Teen', 'Adult', 'Senior'] #set the labels for the bins

# 2. Create the new Age_Category column
df['age_category'] = pd.cut(df['age'], bins=bins, labels=labels)

# 3. Check the results
print(df[['age', 'age_category']].head(10))

# See the count for each category
print(df['age_category'].value_counts())

    age age_category
0  22.0        Adult
1  38.0        Adult
2  26.0        Adult
3  35.0        Adult
4  35.0        Adult
5  28.0        Adult
6  54.0        Adult
7   2.0        Child
8  27.0        Adult
9  14.0         Teen
age_category
Adult     623
Teen       90
Child      68
Senior      0
Name: count, dtype: int64


changed the ranges for bins for senior - where the cap from previous adjustment has knocked out any ages over it previously

In [24]:
bins = [0, 12, 19, 50, 100] #set values for bins (switched 60 to 50 for upper ages)
labels = ['Child', 'Teen', 'Adult', 'Senior'] #set the labels for the bins

# 2. Create the new Age_Category column
df['age_category'] = pd.cut(df['age'], bins=bins, labels=labels)

# 3. Check the results
print(df[['age', 'age_category']].head(10))

# See the count for each category
print(df['age_category'].value_counts())

    age age_category
0  22.0        Adult
1  38.0        Adult
2  26.0        Adult
3  35.0        Adult
4  35.0        Adult
5  28.0        Adult
6  54.0       Senior
7   2.0        Child
8  27.0        Adult
9  14.0         Teen
age_category
Adult     559
Teen       90
Child      68
Senior     64
Name: count, dtype: int64


In [ ]:
##testing ver##
# def get_outliers(df):
#     outlier_report = {} #set empty df for return here
#     get_cols = df.select_dtypes(include=[np.number]).columns
#     for col in get_cols:#get the columns to loop through, where they are numeric type
#         Q1 = df[col].quantile(0.25) 
#         Q3 = df[col].quantile(0.75)
#         IQR = Q3 - Q1
#         lower_bound = Q1 - 1.5 * IQR
#         upper_bound = Q3 + 1.5 * IQR
#         # return lower_bound, upper_bound
#         # print(IQR)
#         outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
#         # print(outliers)
#         outlier_report[col] = {
#             'count': len(outliers),
#             'bounds': (lower_bound, upper_bound),
#             'outlier_values': outliers.tolist()}
    
#     return outlier_report

In [ ]:
def get_outliers(df):
    outlier_report = {} #set empty df for return here
    get_cols = df.select_dtypes(include=[np.number]).columns
    for col in get_cols:#get the columns to loop through, where they are numeric type
        Q1 = df[col].quantile(0.25) 
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]

        outlier_report[col] = {
            'count': len(outliers),
            'bounds': (lower_bound, upper_bound),
            'outlier_values': outliers.tolist()}
    
    return outlier_report

In [51]:
def get_outlier_dfs(df):
    outlier_report = {}
    clean_df = df.copy()  # Create copy for capping
    # Mask to track any row that contains at least one outlier
    all_outliers_mask = pd.Series(False, index=df.index)
    
    get_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in get_cols:
        Q1 = df[col].quantile(0.25) 
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Identify outliers for this specific column
        is_outlier = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers = df[is_outlier][col]
        
        # Update the master mask for "outlier rows"
        all_outliers_mask = all_outliers_mask | is_outlier

        # Requirement 1: Cap values in the clean_df
        clean_df[col] = clean_df[col].clip(lower=lower_bound, upper=upper_bound)

        outlier_report[col] = {
            'count': len(outliers),
            'bounds': (lower_bound, upper_bound),
            'outlier_values': outliers.tolist()}
    
    # Requirement 2: Create DF with outlier rows only
    outlier_df = df[all_outliers_mask]
    
    return outlier_report, clean_df, outlier_df

In [ ]:
report_dict, clean_df, outlier_df = get_outlier_dfs(df)
# report_table = pd.DataFrame.from_dict(report_dict, orient='index')
# print(report_table)
# report_table
clean_df


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,Ticket_Purchase_Date,Purchase_Year,Purchase_Month,age_category
7,0,3,male,2.0,3,1,21.0750,2,Third,child,False,NaN,Southampton,no,False,5,1912-01-15,1912,1,Child
13,0,3,male,39.0,1,5,31.2750,2,Third,man,True,NaN,Southampton,no,False,7,1912-01-15,1912,1,Adult
16,0,3,male,2.0,4,1,29.1250,1,Third,child,False,NaN,Queenstown,no,False,6,1912-01-15,1912,1,Child
24,0,3,female,8.0,3,1,21.0750,2,Third,child,False,NaN,Southampton,no,False,5,1912-01-15,1912,1,Child
25,1,3,female,38.0,1,5,31.3875,2,Third,woman,False,NaN,Southampton,yes,False,7,1912-01-15,1912,1,Adult
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824,0,3,male,2.0,4,1,39.6875,2,Third,child,False,NaN,Southampton,no,False,6,1912-01-15,1912,1,Child
850,0,3,male,4.0,4,2,31.2750,2,Third,child,False,NaN,Southampton,no,False,7,1912-01-15,1912,1,Child
858,1,3,female,24.0,0,3,19.2583,0,Third,woman,False,NaN,Cherbourg,yes,False,4,1912-01-15,1912,1,Adult
885,0,3,female,39.0,0,5,29.1250,1,Third,woman,False,NaN,Queenstown,no,False,6,1912-01-15,1912,1,Adult


In [58]:
outlier_df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,Ticket_Purchase_Date,Purchase_Year,Purchase_Month,age_category
7,0,3,male,2.0,3,1,21.0750,2,Third,child,False,NaN,Southampton,no,False,5,1912-01-15,1912,1,Child
13,0,3,male,39.0,1,5,31.2750,2,Third,man,True,NaN,Southampton,no,False,7,1912-01-15,1912,1,Adult
16,0,3,male,2.0,4,1,29.1250,1,Third,child,False,NaN,Queenstown,no,False,6,1912-01-15,1912,1,Child
24,0,3,female,8.0,3,1,21.0750,2,Third,child,False,NaN,Southampton,no,False,5,1912-01-15,1912,1,Child
25,1,3,female,38.0,1,5,31.3875,2,Third,woman,False,NaN,Southampton,yes,False,7,1912-01-15,1912,1,Adult
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824,0,3,male,2.0,4,1,39.6875,2,Third,child,False,NaN,Southampton,no,False,6,1912-01-15,1912,1,Child
850,0,3,male,4.0,4,2,31.2750,2,Third,child,False,NaN,Southampton,no,False,7,1912-01-15,1912,1,Child
858,1,3,female,24.0,0,3,19.2583,0,Third,woman,False,NaN,Cherbourg,yes,False,4,1912-01-15,1912,1,Adult
885,0,3,female,39.0,0,5,29.1250,1,Third,woman,False,NaN,Queenstown,no,False,6,1912-01-15,1912,1,Adult


In [ ]:
# test = get_outliers(df)
# print(test) #hard to read, need to append into a df

{'survived': {'count': 0, 'bounds': (-1.5, 2.5), 'outlier_values': []}, 'pclass': {'count': 0, 'bounds': (-2.0, 6.0), 'outlier_values': []}, 'age': {'count': 0, 'bounds': (1.0, 57.0), 'outlier_values': []}, 'sibsp': {'count': 39, 'bounds': (-1.5, 2.5), 'outlier_values': [3, 4, 3, 3, 4, 5, 3, 4, 5, 3, 3, 4, 8, 4, 4, 3, 8, 4, 3, 4, 4, 4, 4, 3, 3, 5, 5, 4, 4, 3, 3, 5, 4, 3, 4, 4, 3, 4, 4]}, 'parch': {'count': 15, 'bounds': (-1.5, 2.5), 'outlier_values': [5, 5, 3, 4, 4, 3, 4, 4, 5, 5, 6, 3, 3, 3, 5]}, 'fare': {'count': 0, 'bounds': (-30.906200000000002, 72.977), 'outlier_values': []}, 'embarked': {'count': 0, 'bounds': (-0.5, 3.5), 'outlier_values': []}, 'family_size': {'count': 83, 'bounds': (-0.5, 3.5), 'outlier_values': [5, 7, 6, 5, 7, 6, 4, 6, 4, 8, 6, 7, 8, 4, 5, 6, 4, 7, 5, 11, 6, 6, 6, 5, 11, 7, 4, 5, 7, 7, 6, 6, 4, 4, 5, 6, 6, 5, 8, 4, 4, 5, 6, 6, 4, 4, 4, 4, 8, 4, 4, 7, 7, 5, 4, 4, 7, 4, 4, 6, 6, 6, 8, 8, 4, 6, 4, 5, 5, 4, 4, 5, 4, 6, 4, 4, 7, 6, 6, 7, 4, 6, 4]}, 'Purchase_Year': 

**Final Ver outliers func**

In [ ]:
def get_outliers(df):
    clean_df = df.copy()
    # Create a mask to track any row that has an outlier in any column
    outlier_mask = pd.Series(False, index=df.index)
    
    get_cols = df.select_dtypes(include=[np.number]).columns #onyl get numeric cols
    
    for col in get_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Identify outliers for this column
        is_outlier = (df[col] < lower_bound) | (df[col] > upper_bound)
        
        # Update the master mask
        outlier_mask = outlier_mask | is_outlier
        
        # Update clean_df by capping values
        clean_df[col] = clean_df[col].clip(lower=lower_bound, upper=upper_bound)
    
    # Create the dataframe containing ONLY the original outlier rows
    outlier_df = df[outlier_mask]
    
    return clean_df, outlier_df


In [63]:
clean_df, outlier_df = get_outliers(df)
display(clean_df)
display(outlier_df)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,Ticket_Purchase_Date,Purchase_Year,Purchase_Month,age_category
0,0,3,male,22.0,1.0,0.0,7.2500,2,Third,man,True,NaN,Southampton,no,False,2.0,1912-01-15,1912,1,Adult
1,1,1,female,38.0,1.0,0.0,71.2833,0,First,woman,False,C,Cherbourg,yes,False,2.0,1912-01-15,1912,1,Adult
2,1,3,female,26.0,0.0,0.0,7.9250,2,Third,woman,False,NaN,Southampton,yes,True,1.0,1912-01-15,1912,1,Adult
3,1,1,female,35.0,1.0,0.0,53.1000,2,First,woman,False,C,Southampton,yes,False,2.0,1912-01-15,1912,1,Adult
4,0,3,male,35.0,0.0,0.0,8.0500,2,Third,man,True,NaN,Southampton,no,True,1.0,1912-01-15,1912,1,Adult
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,0,3,female,39.0,0.0,2.5,29.1250,1,Third,woman,False,NaN,Queenstown,no,False,3.5,1912-01-15,1912,1,Adult
887,1,1,female,19.0,0.0,0.0,30.0000,2,First,woman,False,B,Southampton,yes,True,1.0,1912-01-15,1912,1,Teen
888,0,3,female,28.0,1.0,2.0,23.4500,2,Third,woman,False,NaN,Southampton,no,False,3.5,1912-01-15,1912,1,Adult
889,1,1,male,26.0,0.0,0.0,30.0000,0,First,man,True,C,Cherbourg,yes,True,1.0,1912-01-15,1912,1,Adult


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,Ticket_Purchase_Date,Purchase_Year,Purchase_Month,age_category
7,0,3,male,2.0,3,1,21.0750,2,Third,child,False,NaN,Southampton,no,False,5,1912-01-15,1912,1,Child
13,0,3,male,39.0,1,5,31.2750,2,Third,man,True,NaN,Southampton,no,False,7,1912-01-15,1912,1,Adult
16,0,3,male,2.0,4,1,29.1250,1,Third,child,False,NaN,Queenstown,no,False,6,1912-01-15,1912,1,Child
24,0,3,female,8.0,3,1,21.0750,2,Third,child,False,NaN,Southampton,no,False,5,1912-01-15,1912,1,Child
25,1,3,female,38.0,1,5,31.3875,2,Third,woman,False,NaN,Southampton,yes,False,7,1912-01-15,1912,1,Adult
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
824,0,3,male,2.0,4,1,39.6875,2,Third,child,False,NaN,Southampton,no,False,6,1912-01-15,1912,1,Child
850,0,3,male,4.0,4,2,31.2750,2,Third,child,False,NaN,Southampton,no,False,7,1912-01-15,1912,1,Child
858,1,3,female,24.0,0,3,19.2583,0,Third,woman,False,NaN,Cherbourg,yes,False,4,1912-01-15,1912,1,Adult
885,0,3,female,39.0,0,5,29.1250,1,Third,woman,False,NaN,Queenstown,no,False,6,1912-01-15,1912,1,Adult


In [ ]:
# test = get_outliers(df)
# test2 = pd.DataFrame.from_dict(test, orient='index') #flip to list columns as rows
# test2

,count,bounds,outlier_values
survived,0,"(-1.5, 2.5)",[]
pclass,0,"(-2.0, 6.0)",[]
age,0,"(1.0, 57.0)",[]
sibsp,39,"(-1.5, 2.5)","[3, 4, 3, 3, 4, 5, 3, 4, 5, 3, 3, 4, 8, 4, 4, ..."
parch,15,"(-1.5, 2.5)","[5, 5, 3, 4, 4, 3, 4, 4, 5, 5, 6, 3, 3, 3, 5]"
fare,0,"(-30.906200000000002, 72.977)",[]
embarked,0,"(-0.5, 3.5)",[]
family_size,83,"(-0.5, 3.5)","[5, 7, 6, 5, 7, 6, 4, 6, 4, 8, 6, 7, 8, 4, 5, ..."
Purchase_Year,0,"(1912.0, 1912.0)",[]
Purchase_Month,0,"(1.0, 1.0)",[]


**duplicate & na rows func**

In [46]:
def dedupe_na(df): # only need dataframe to set
    return df.drop_duplicates().dropna()

In [64]:
dupenatest = dedupe_na(df)
dupenatest

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,Ticket_Purchase_Date,Purchase_Year,Purchase_Month,age_category
1,1,1,female,38.0,1,0,71.2833,0,First,woman,False,C,Cherbourg,yes,False,2,1912-01-15,1912,1,Adult
3,1,1,female,35.0,1,0,53.1000,2,First,woman,False,C,Southampton,yes,False,2,1912-01-15,1912,1,Adult
6,0,1,male,54.0,0,0,51.8625,2,First,man,True,E,Southampton,no,True,1,1912-01-15,1912,1,Senior
10,1,3,female,4.0,1,1,16.7000,2,Third,child,False,G,Southampton,yes,False,3,1912-01-15,1912,1,Child
11,1,1,female,57.0,0,0,26.5500,2,First,woman,False,C,Southampton,yes,True,1,1912-01-15,1912,1,Senior
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
871,1,1,female,47.0,1,1,52.5542,2,First,woman,False,D,Southampton,yes,False,3,1912-01-15,1912,1,Adult
872,0,1,male,33.0,0,0,5.0000,2,First,man,True,B,Southampton,no,True,1,1912-01-15,1912,1,Adult
879,1,1,female,56.0,0,1,72.9770,0,First,woman,False,C,Cherbourg,yes,False,2,1912-01-15,1912,1,Senior
887,1,1,female,19.0,0,0,30.0000,2,First,woman,False,B,Southampton,yes,True,1,1912-01-15,1912,1,Teen


**test on another dataset**

In [70]:
dfp = sns.load_dataset("penguins")
# dfp.head()
dfp

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [68]:
clean_dfp, outlier_dfp = get_outliers(dfp)
display(clean_dfp)
display(outlier_dfp)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex


In [69]:
dupenap = dedupe_na(dfp)
dupenap

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female
